In [ ]:
# Homogenization Algorithm
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter
from skimage.segmentation import slic
from scipy.interpolate import interp1d

# ========== Image Enhancement Module ==========
def plot_histogram(image, title):
    """Visualize histogram enhancement"""
    plt.hist(image.ravel(), bins=256, range=[0, 256])
    plt.title(title)
    plt.xlabel('Gray Value')
    plt.ylabel('Frequency')
    plt.xlim([0, 256])

def enhanced_double_histogram_equalization(image, gamma=1.2, clip_limit=2.0, tile_grid_size=(8,8)):
    """Dynamic tri-partition histogram equalization"""
    """
    Improved tri-partition histogram equalization method.

    Parameters:
        image (numpy.ndarray): Input image, either grayscale or color.
        gamma (float): Gamma correction parameter for adjusting image brightness and contrast.
        clip_limit (float): CLAHE contrast limiting parameter used to control enhancement strength.
        tile_grid_size (tuple): CLAHE grid size for dividing the image into local regions.

    Returns:
        numpy.ndarray: Image processed by the improved double histogram equalization method.
    """
    # If the input image is color, first convert it to grayscale
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image.copy()

    # Gamma correction to adjust image brightness and contrast
    gray = np.power(gray / 255.0, gamma) * 255
    gray = gray.astype(np.uint8)

    # Compute grayscale histogram and cumulative distribution function
    hist, bins = np.histogram(gray.flatten(), 256, [0, 256])
    cum_hist = hist.cumsum()
    total_pixels = cum_hist[-1]

    # Dynamically divide brightness regions to determine thresholds for low, mid, and high brightness
    low_threshold = np.searchsorted(cum_hist, 0.3 * total_pixels)   # 30th percentile
    high_threshold = np.searchsorted(cum_hist, 0.8 * total_pixels)  # 80th percentile

    # Generate masks for brightness regions
    low_mask = gray < low_threshold
    mid_mask = (gray >= low_threshold) & (gray < high_threshold)
    high_mask = gray >= high_threshold

    # Create CLAHE object
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    # Process low-brightness region: enhance contrast using CLAHE
    low_region = clahe.apply(gray.copy())
    low_region[~low_mask] = 0  # Set non-low-brightness regions to 0

    # Process mid-brightness region: CLAHE equalization with larger block size
    mid_region = clahe.apply(gray.copy())
    mid_region[~mid_mask] = 0  # Set non-mid-brightness regions to 0

    # Process high-brightness region: use CLAHE to suppress overexposure
    high_region = clahe.apply(gray.copy())
    high_region[~high_mask] = 0  # Set non-high-brightness regions to 0

    # Merge the results from the three regions
    merged = low_region + mid_region + high_region

    return merged.astype(np.uint8)

def compute_integral_image(image):
    """Optimized integral image computation"""
    h, w = image.shape
    integral = np.zeros((h + 1, w + 1), dtype=np.float32)

    for i in range(1, h + 1):
        row_sum = 0
        for j in range(1, w + 1):
            row_sum += image[i - 1, j - 1]
            integral[i, j] = integral[i - 1, j] + row_sum

    return integral

def get_patch_sum(integral, x1, y1, x2, y2):
    """
    Quickly compute the sum of a rectangular region using the integral image.
    """
    return integral[x2 + 1, y2 + 1] - integral[x1, y2 + 1] - integral[x2 + 1, y1] + integral[x1, y1]

def improved_fast_nlm_denoise(image, search_window=15, similarity_window=9, sigma=3.0):
    """Fast non-local means denoising"""
    """
    Improved fast non-local means denoising algorithm.
    """
    h, w = image.shape
    N = similarity_window
    S = search_window

    integral = compute_integral_image(image)
    denoised = np.zeros_like(image, dtype=np.float32)

    for i in range(h):
        for j in range(w):
            # Define the search window
            patch_start_i = max(0, i - S // 2)
            patch_end_i = min(h, i + S // 2 + 1)
            patch_start_j = max(0, j - S // 2)
            patch_end_j = min(w, j + S // 2 + 1)

            # Position of the center patch
            center_start_i = i - N // 2
            center_end_i = i + N // 2 + 1
            center_start_j = j - N // 2
            center_end_j = j + N // 2 + 1

            # Ensure the center patch is within image bounds
            center_start_i = max(0, center_start_i)
            center_end_i = min(h, center_end_i)
            center_start_j = max(0, center_start_j)
            center_end_j = min(w, center_end_j)

            # Integral value of the center patch
            center_val = get_patch_sum(
                integral,
                center_start_i, center_start_j,
                center_end_i - 1, center_end_j - 1
            )

            total_weight = 0.0
            weighted_sum = 0.0

            # Traverse all possible patches in the search window
            for di in range(patch_start_i, patch_end_i):
                for dj in range(patch_start_j, patch_end_j):
                    # Position of the current comparison patch
                    comp_start_i = di - N // 2
                    comp_end_i = di + N // 2 + 1
                    comp_start_j = dj - N // 2
                    comp_end_j = dj + N // 2 + 1

                    # Ensure the comparison patch is within image bounds
                    comp_start_i = max(0, comp_start_i)
                    comp_end_i = min(h, comp_end_i)
                    comp_start_j = max(0, comp_start_j)
                    comp_end_j = min(w, comp_end_j)

                    # Compute similarity between the current patch and the center patch
                    comp_val = get_patch_sum(
                        integral,
                        comp_start_i, comp_start_j,
                        comp_end_i - 1, comp_end_j - 1
                    )

                    # Compute difference (simplified as absolute difference)
                    diff = abs(center_val - comp_val)
                    weight = np.exp(-diff / (sigma ** 2))

                    # Update total weight and weighted sum
                    total_weight += weight
                    weighted_sum += weight * image[di, dj]

            if total_weight == 0:
                denoised[i, j] = image[i, j]
            else:
                denoised[i, j] = weighted_sum / total_weight
    return denoised.astype(np.uint8)

# ========== Improved Tamura Coarseness Calculation ==========
def tamura_coarseness_v2(image, max_scale=6, contrast_threshold=0.03):
    """
    Improved Tamura coarseness calculation
    Optimization points: hybrid multi-scale analysis, spatial consistency constraint,
    and subpixel interpolation
    """
    # ===== Preprocessing =====
    img_original = image.astype(np.float32)
    h, w = img_original.shape

    max_pad = int(1.5 * 2**max_scale)
    img = cv2.copyMakeBorder(
        img_original, max_pad, max_pad, max_pad, max_pad,
        cv2.BORDER_REFLECT101
    )
    h_pad, w_pad = img.shape

    # ===== Hybrid multi-scale generation =====
    scales = []
    base = 2
    while base < 2**max_scale:
        scales.append(base)
        if base < 8:
            scales.append(base + 1)
            base *= 2
        else:
            base += 3
        if 2 * base > min(h_pad, w_pad) // 2:
            break
    scales = np.unique(np.array(scales))

    # ===== Multi-scale energy calculation =====
    energy_stack = []
    for s in scales:
        win_size = 2 * s + 1
        mean_map = uniform_filter(img, size=win_size, mode='reflect')

        diff = np.abs(img - mean_map)
        sigma = max(0.3 * s, 0.7)
        blurred_diff = cv2.GaussianBlur(diff, (win_size, win_size), sigmaX=sigma)
        energy_stack.append(blurred_diff)

    energy_stack = np.array(energy_stack)
    best_scales_idx = np.argmax(energy_stack, axis=0)

    # ===== Subpixel interpolation =====
    refined_scales = np.zeros_like(best_scales_idx, dtype=np.float32)
    x = np.arange(len(scales))
    for i in range(h_pad):
        for j in range(w_pad):
            idx = best_scales_idx[i, j]
            if idx == 0 or idx == len(scales) - 1:
                refined_scales[i, j] = scales[idx]
                continue

            y_vals = energy_stack[idx - 1:idx + 2, i, j]
            f = interp1d([-1, 0, 1], y_vals, kind='quadratic')
            x_new = np.linspace(-0.5, 0.5, 3)
            max_pos = x_new[np.argmax(f(x_new))]
            refined_scales[i, j] = scales[idx] + max_pos * (scales[idx + 1] - scales[idx])

    # ===== Adaptive threshold =====
    global_energy = np.max(energy_stack, axis=0)
    local_contrast = cv2.boxFilter(global_energy, -1, (31, 31))
    adaptive_thresh = contrast_threshold * cv2.GaussianBlur(local_contrast, (51, 51), 0)
    mask = (global_energy > adaptive_thresh)

    # ===== Result calculation =====
    valid_area = refined_scales[max_pad:-max_pad, max_pad:-max_pad][
        mask[max_pad:-max_pad, max_pad:-max_pad]
    ]
    coarseness = np.mean(valid_area) * 0.1

    # ===== Original heatmap generation =====
    heatmap = refined_scales[max_pad:-max_pad, max_pad:-max_pad]  # Directly crop the valid region
    heatmap_enhanced = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX)  # Normalize to 0-255

    return coarseness, cv2.applyColorMap(heatmap_enhanced.astype(np.uint8), cv2.COLORMAP_JET)

# ========== Full Processing Pipeline ==========
def full_processing_pipeline(image_path):
    """End-to-end image analysis pipeline"""
    # Read image
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError("Unable to read the image. Please check the file path.")

    # Preprocessing stage
    print("[Stage 1] Image enhancement processing...")
    dhe_enhanced = enhanced_double_histogram_equalization(image)
    denoised = improved_fast_nlm_denoise(
        dhe_enhanced,
        search_window=11,
        similarity_window=7,
        sigma=1.0
    )

    # Feature analysis stage
    print("[Stage 2] Texture feature calculation...")
    coarseness, heatmap = tamura_coarseness_v2(
        denoised,
        max_scale=6,
        contrast_threshold=0.03
    )

    # Result visualization
    print("[Stage 3] Visualization output...")
    plt.figure(figsize=(18, 12))

    # Original image information
    plt.subplot(3, 4, 1)
    plt.imshow(image, cmap='gray')
    plt.title('Original Image\nScale: {}'.format(image.shape))
    plt.axis('off')

    plt.subplot(3, 4, 2)
    plot_histogram(image, 'Original Histogram')

    # Enhancement effect
    plt.subplot(3, 4, 3)
    plt.imshow(dhe_enhanced, cmap='gray')
    plt.title('Tri-Zone Enhanced\nContrast Enhancement: {:.1f}%'.format(
        (np.std(dhe_enhanced) / np.std(image) - 1) * 100))
    plt.axis('off')

    plt.subplot(3, 4, 4)
    plot_histogram(dhe_enhanced, 'Tri-Zone Enhanced Histogram')

    # Denoising effect
    plt.subplot(3, 4, 5)
    plt.imshow(denoised, cmap='gray')
    plt.title('NLM Denoising\nPSNR: {:.2f} dB'.format(
        10 * np.log10(255**2 / np.mean((denoised.astype(float) - image.astype(float))**2))))
    plt.axis('off')

    plt.subplot(3, 4, 6)
    plot_histogram(denoised, 'Denoised Image Histogram')

    # Coarseness analysis
    plt.subplot(3, 4, 7)
    plt.imshow(heatmap)
    plt.title('Improved Coarseness Value (ICV): {:.2f} μm'.format(coarseness))
    plt.axis('off')

    plt.tight_layout(pad=3.0)
    plt.show()

# ========== Execution Example ==========
if __name__ == "__main__":
    image_path = r'C:\light_cv_project\data\new25527\5\7v1A\LDNMT_004.tiff'
    try:
        full_processing_pipeline(image_path)
    except Exception as e:
        print(f"Processing failed: {str(e)}")
